# Conversion Prediction Model

# Objective
**Problem:** Predict which free-tier (non-customer) companies are likely to convert to paying customers_df_df_df_df_df_df_df_df_df within the next 30 days.

**Output:** Weekly prioritized list (generated on Sundays).

# Assumptions
+ Alexa rank Null means that the company has no webpage, thus we assign 16000001 to these cases

# Table of Contents

In [135]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, IFrame
from ydata_profiling import ProfileReport
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
def missing_summary(df, name):
    """
    Prints a summary of columns in the DataFrame `df` that contain missing values,
    including the count and percentage of missing entries for each column.
    
    Args:
        df (pd.DataFrame): The DataFrame to analyze.
        name (str): Label or name to use when reporting missing columns.
    
    Output:
        Prints a formatted list of columns with missing values to the console.
    """
    miss = df.isna().sum()
    miss = miss[miss > 0]

    print(f"\n{name} columns with missing values:")
    if miss.empty:
        print("None")
        return

    out = pd.DataFrame({
        "column": miss.index,
        "missing_count": miss.values,
        "missing_pct": (miss.values / len(df) * 100).round(2),
    }).sort_values(["missing_pct", "missing_count"], ascending=False)

    print(out.to_string(index=False))


def encode_industries(df, column='INDUSTRY', max_cats=10):
    """
    Improved industry encoder with refined regex, standardized naming, 
    and robust error handling.
    """
    # 1. Cleaning & Normalization
    # Replace separators with spaces to help regex find word boundaries
    series = (df[column].fillna('UNKNOWN')
              .astype(str)
              .str.upper()
              .str.replace(r'[/_\-&]', ' ', regex=True)
              .str.strip())
    
    # 2. Refined Strategic Mapping
    # Ordered from most specific to most general to avoid misclassification
    mapping = {
        r'.*(SOFTWARE|SAAS|TECH).*': 'TECH',
        r'.*(IT|INFORMATION TECHNOLOGY|COMPUTER|NETWORK).*': 'IT_SERVICES',
        r'.*(FINANCE|BANKING|ACCOUNTING|INSURANCE|INVEST).*': 'FINANCE',
        r'.*(CONSULTING|ADVISORY|STRATEGY).*': 'CONSULTING',
        r'.*(STAFFING|RECRUIT|HR|HUMAN RESOURCE).*': 'HR_STAFFING',
        r'.*(EDUCATION|LEARNING|SCHOOL|UNIVERSITY|ACADEMIC).*': 'EDUCATION',
        r'.*(MARKETING|ADVERTISING|PR|PUBLIC RELATIONS).*': 'MARKETING',
        r'.*(ENERGY|RENEWABLE|UTILITY|OIL|GAS).*': 'ENERGY_UTILITIES',
        r'.*(HEALTH|MEDICAL|PHARMA|HOSPITAL).*': 'HEALTHCARE',
        r'.*(NON PROFIT|CIVIC|RELIGIOUS|GOV).*': 'NON_PROFIT'
    }
    
    # Apply mapping
    for pattern, label in mapping.items():
        series = series.replace(pattern, label, regex=True)
    
    # 3. One-Hot Encoding
    encoder = OneHotEncoder(
        max_categories=max_cats,
        handle_unknown='infrequent_if_exist',
        sparse_output=False
    )
    
    # Fit-Transform and create DataFrame
    encoded_array = encoder.fit_transform(series.to_frame())
    feature_names = [f"IND_{c.split('_')[-1]}" for c in encoder.get_feature_names_out([column])]
    
    ohe_df = pd.DataFrame(
        encoded_array, 
        columns=feature_names, 
        index=df.index
    ).astype(int)
    
    return pd.concat([df, ohe_df], axis=1)

# Constants

In [ ]:
ACTION_COLS = ["ACTIONS_CRM_CONTACTS", "ACTIONS_CRM_COMPANIES", "ACTIONS_CRM_DEALS", "ACTIONS_EMAIL"]
USER_COLS   = ["USERS_CRM_CONTACTS", "USERS_CRM_COMPANIES", "USERS_CRM_DEALS", "USERS_EMAIL"]
METRIC_COLS = ACTION_COLS + USER_COLS
WINDOWS     = {"7d": 7, "14d": 14, "30d": 30, "60d": 60}
PREDICTION_HORIZON_DAYS = 30

# Load Data
- Load data
- Preprocess and standardize columns
- Check data integrity and basic statistics

In [136]:

# Load data
customers_df = pd.read_csv("../data/customers.csv")
noncustomers_df = pd.read_csv("../data/noncustomers.csv")
usage_actions_df = pd.read_csv("../data/usage_actions.csv")

# Parse dates
customers_df['CLOSEDATE'] = pd.to_datetime(customers_df['CLOSEDATE'])
usage_actions_df['WHEN_TIMESTAMP'] = pd.to_datetime(usage_actions_df['WHEN_TIMESTAMP'])

# Sort by CLOSEDATE and WHEN_TIMESTAMP
customers_df = customers_df.sort_values(by=['CLOSEDATE'])
usage_actions_df = usage_actions_df.sort_values(by=['WHEN_TIMESTAMP'])

# All columns to upper case
customers_df.columns = customers_df.columns.str.upper()
noncustomers_df.columns = noncustomers_df.columns.str.upper()
usage_actions_df.columns = usage_actions_df.columns.str.upper()


# Check data
print(f"customers:     {customers_df.shape}  — IDs {customers_df['ID'].min()}–{customers_df['ID'].max()}")
print(f"noncustomers:  {noncustomers_df.shape}  — IDs {noncustomers_df['ID'].min()}–{noncustomers_df['ID'].max()}")
print(f"usage_actions: {usage_actions_df.shape}")
print(f"Usage date range: {usage_actions_df['WHEN_TIMESTAMP'].min().date()} → {usage_actions_df['WHEN_TIMESTAMP'].max().date()}")
print(f"Unique ids: {usage_actions_df['ID'].nunique()}")

cust_ids   = set(customers_df["ID"])
noncust_ids= set(noncustomers_df["ID"])
print(f"ID overlap (customers ∩ noncustomers): {len(cust_ids & noncust_ids)}")  

REF_DATE = usage_actions_df["WHEN_TIMESTAMP"].max()
print(f"Reference date ('today'): {REF_DATE.date()}")


customers:     (200, 6)  — IDs 1–200
noncustomers:  (5003, 4)  — IDs 201–5200
usage_actions: (25387, 10)
Usage date range: 2019-01-07 → 2020-07-27
Unique ids: 3569
ID overlap (customers ∩ noncustomers): 0
Reference date ('today'): 2020-07-27


In [ ]:
# Usage-level comparison: customers vs non-customers
historical_customer_ids = set(historical_customers_df["ID"])
usage_customers_df = usage_actions_df[usage_actions_df["ID"].isin(historical_customer_ids)].copy()
usage_noncustomers_df = usage_actions_df[usage_actions_df["ID"].isin(noncust_ids)].copy()

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

usage_customer_profile = ProfileReport(usage_customers_df, title="Customer Usage", minimal=True)
usage_noncustomer_profile = ProfileReport(usage_noncustomers_df, title="Non-customer Usage", minimal=True)

usage_customer_profile.compare(usage_noncustomer_profile).to_file(reports_dir / "usage_comparison.html")

display(IFrame(src=str(reports_dir / "usage_comparison.html"), width="100%", height=600))

# EDA
+ Missing values
+ Using ydata_profiling for detailed and shareable insights found in ./reports

In [ ]:
# Check missing data
missing_summary(customers_df, "CUSTOMERS")
missing_summary(noncustomers_df, "NON-CUSTOMERS")
missing_summary(usage_actions_df, "USAGE")



CUSTOMERS columns with missing values:
        column  missing_count  missing_pct
      INDUSTRY            129         64.5
EMPLOYEE_RANGE              2          1.0

NON-CUSTOMERS columns with missing values:
        column  missing_count  missing_pct
      INDUSTRY           3725        74.46
EMPLOYEE_RANGE            532        10.63
    ALEXA_RANK            114         2.28

USAGE columns with missing values:
None


In [138]:
# Company-level comparison: customers vs non-customers
historical_customers_df = customers_df[customers_df["CLOSEDATE"] <= REF_DATE].copy()

reports_dir = Path("../reports")
reports_dir.mkdir(parents=True, exist_ok=True)

customer_profile    = ProfileReport(historical_customers_df, title="Customers",     minimal=True)
noncustomer_profile = ProfileReport(noncustomers_df,         title="Non-customers", minimal=True)

customer_profile.compare(noncustomer_profile).to_file(reports_dir / "company_comparison.html")

display(IFrame(src=str(reports_dir / "company_comparison.html"), width="100%", height=600))

c:\Users\totic\anaconda3\Lib\site-packages\ydata_profiling\compare_reports.py:191: UserWarning: The datasets being profiled have a different set of columns. Only the left side profile will be calculated.
  warnings.warn(


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 6/6 [00:00<00:00, 993.60it/s]


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 4/4 [00:00<00:00, 301.94it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

# Feature Engineering
+ Assumption: Alexa rank Null means that the company has no webpage, thus we assign 16000001 to these cases
+ INDUSTRY nulls could be reduced by doing Web Scraping or buying a company industry database
+ Employee range to the middle of the range
+ One Hot Encoding of INDUSTRY
+ Build date features

In [140]:
# 1. Add Label and Align Columns
# We use assign to create the label without modifying the original dfs immediately
customers_base = customers_df.assign(CUSTOMER=1)
noncustomers_base = noncustomers_df.assign(CUSTOMER=0)

# 2. Canonical Schema Alignment
# Ensure non-customers have all columns from customers (filled with NaN where missing)
target_cols = customers_base.columns
noncustomers_base = noncustomers_base.reindex(columns=target_cols)

# 3. Bulk Type Alignment
# Map the dtypes from customers directly onto non-customers in one go.
# We convert both to 'nullable' types first to prevent NaN-casting issues.
companies_df = pd.concat([customers_base, noncustomers_base], axis=0, ignore_index=True)

# Force specific critical types (like Datetime) if convert_dtypes is too conservative
companies_df['CLOSEDATE'] = pd.to_datetime(companies_df['CLOSEDATE'], errors='coerce')

print(f"Merged companies_df shape: {companies_df.shape}")
print(companies_df["CUSTOMER"].value_counts(dropna=False))

Merged companies_df shape: (5203, 7)
CUSTOMER
0    5003
1     200
Name: count, dtype: int64


C:\Users\totic\AppData\Local\Temp\ipykernel_48680\3376098774.py:14: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  companies_df = pd.concat([customers_base, noncustomers_base], axis=0, ignore_index=True)


In [141]:
# For null ALEXA_RANK assume they where not found and page-rank will be weak
max_rank = companies_df['ALEXA_RANK'].max()
companies_df['ALEXA_RANK'] = np.where(companies_df['ALEXA_RANK'].isnull(), max_rank, companies_df['ALEXA_RANK'])

In [142]:
# Employee range to numberic
# For a better comparison, map EMPLOYEE_RANGE with explicit dictionaries.
EMPLOYEE_RANGE_TO_MID = {
    "1": 1.0,
    "2 to 5": 3.0,
    "6 to 10": 8.0,
    "11 to 25": 18.0,
    "26 to 50": 38.0,
    "51 to 200": 125.0,
    "201 to 1000": 600.0,
    "1001 to 10000": 5500.0,
    "10,001 or more": 10001.0,
}

companies_df["EMPLOYEE_RANGE"] = companies_df["EMPLOYEE_RANGE"].map(EMPLOYEE_RANGE_TO_MID)


In [ ]:
# Encoding Insustry
companies_df = encode_industries(companies_df, 'INDUSTRY', max_cats=10)
companies_df

,CLOSEDATE,MRR,ALEXA_RANK,EMPLOYEE_RANGE,INDUSTRY,ID,CUSTOMER,IND_CONSULTING,IND_EDUCATION,IND_FINANCE,IND_SERVICES,IND_MARKETING,IND_OTHER,IND_REAL ESTATE,IND_TECH,IND_UNKNOWN,IND_sklearn
0,2019-01-15,300.96,16000001.0,8.0,NaN,42,1,0,0,0,0,0,0,0,0,1,0
1,2019-01-17,49.07,10390859.0,3.0,NaN,20,1,0,0,0,0,0,0,0,0,1,0
2,2019-01-27,209.98,273063.0,38.0,Technology - Software,174,1,0,0,0,0,0,0,0,1,0,0
3,2019-01-30,40.00,2610402.0,3.0,RENEWABLES_ENVIRONMENT,1,1,0,0,0,0,0,0,0,0,0,1
4,2019-01-30,400.00,16000001.0,18.0,NaN,25,1,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5198,NaT,NaN,16000001.0,NaN,NaN,637,0,0,0,0,0,0,0,0,0,1,0
5199,NaT,NaN,20183.0,5500.0,Non-Profit/Educational Institution,4921,0,0,0,0,1,0,0,0,0,0,0
5200,NaT,NaN,16000001.0,8.0,NaN,1215,0,0,0,0,0,0,0,0,0,1,0
5201,NaT,NaN,16000001.0,18.0,NaN,2693,0,0,0,0,0,0,0,0,0,1,0


In [44]:
usage_actions_df

,WHEN_TIMESTAMP,ACTIONS_CRM_CONTACTS,ACTIONS_CRM_COMPANIES,ACTIONS_CRM_DEALS,ACTIONS_EMAIL,USERS_CRM_CONTACTS,USERS_CRM_COMPANIES,USERS_CRM_DEALS,USERS_EMAIL,ID
2287,2019-01-07,0,0,0,0,0,0,0,0,4916
2346,2019-01-07,0,0,0,0,0,0,0,0,4432
12713,2019-01-07,136,0,1,0,11,0,1,0,157
12827,2019-01-07,1,1,19,0,1,1,1,0,162
13124,2019-01-07,1,0,0,0,1,0,0,0,4412
...,...,...,...,...,...,...,...,...,...,...
7000,2020-07-27,121,343,94,0,9,14,11,0,51
12384,2020-07-27,1,0,0,0,1,0,0,0,3041
20639,2020-07-27,166,0,106,1,7,0,6,1,32
12307,2020-07-27,22,0,3,0,2,0,1,0,4834


In [146]:
ACTION_COLS = ["ACTIONS_CRM_CONTACTS","ACTIONS_CRM_COMPANIES","ACTIONS_CRM_DEALS","ACTIONS_EMAIL"]
USER_COLS   = ["USERS_CRM_CONTACTS","USERS_CRM_COMPANIES","USERS_CRM_DEALS","USERS_EMAIL"]
METRIC_COLS = ACTION_COLS + USER_COLS
WINDOWS     = {"7d": 7, "14d": 14, "30d": 30, "60d": 60}


def build_usage_features(portal_ids, cutoff_dates_map=None):
    """
    Build time-windowed usage features for a set of portal IDs.
    
    cutoff_dates_map: dict {id -> cutoff_date}. If None, uses REF_DATE for all.
    This is critical to avoid leakage for customers (don't use post-conversion usage).
    """
    all_rows = []
    
    # Filter usage to relevant IDs first for speed
    relevant_usage = usage_actions_df[usage_actions_df["ID"].isin(portal_ids)].copy()
    
    for pid in portal_ids:
        cutoff = cutoff_dates_map.get(pid, REF_DATE) if cutoff_dates_map else REF_DATE
        portal_usage = relevant_usage[
            (relevant_usage["ID"] == pid) & 
            (relevant_usage["WHEN_TIMESTAMP"] < cutoff)
        ]
        
        row = {"ID": pid}
        
        # Window-based features
        for wname, wdays in WINDOWS.items():
            win_start = cutoff - pd.Timedelta(days=wdays)
            w_data = portal_usage[portal_usage["WHEN_TIMESTAMP"] >= win_start]
            for col in METRIC_COLS:
                row[f"{col}_sum_{wname}"]  = w_data[col].sum()
                row[f"{col}_mean_{wname}"] = w_data[col].mean() if len(w_data) > 0 else 0
            row[f"active_days_{wname}"]    = w_data["WHEN_TIMESTAMP"].nunique()
            row[f"total_actions_{wname}"]  = w_data[ACTION_COLS].sum().sum()
            row[f"total_users_{wname}"]    = w_data[USER_COLS].sum().sum()
        
        # Lifetime features
        row["total_actions_lifetime"]     = portal_usage[ACTION_COLS].sum().sum()
        row["total_users_lifetime"]       = portal_usage[USER_COLS].sum().sum()
        row["active_days_lifetime"]       = portal_usage["WHEN_TIMESTAMP"].nunique()
        
        if len(portal_usage) > 0:
            row["days_since_first_usage"] = (cutoff - portal_usage["WHEN_TIMESTAMP"].min()).days
            row["days_since_last_usage"]  = (cutoff - portal_usage["WHEN_TIMESTAMP"].max()).days
            row["usage_recency_score"]    = 1.0 / (row["days_since_last_usage"] + 1)
            # Trend: ratio of last-30d actions to days 31-60
            last30  = portal_usage[portal_usage["WHEN_TIMESTAMP"] >= cutoff - pd.Timedelta(30, "D")][ACTION_COLS].sum().sum()
            prev30  = portal_usage[
                (portal_usage["WHEN_TIMESTAMP"] >= cutoff - pd.Timedelta(60, "D")) &
                (portal_usage["WHEN_TIMESTAMP"] < cutoff - pd.Timedelta(30, "D"))
            ][ACTION_COLS].sum().sum()
            row["action_trend_ratio"] = last30 / (prev30 + 1)  # +1 avoid div by zero
            # Module diversity: how many object types used
            row["module_diversity"] = sum([
                1 if portal_usage["ACTIONS_CRM_CONTACTS"].sum() > 0 else 0,
                1 if portal_usage["ACTIONS_CRM_COMPANIES"].sum() > 0 else 0,
                1 if portal_usage["ACTIONS_CRM_DEALS"].sum() > 0 else 0,
                1 if portal_usage["ACTIONS_EMAIL"].sum() > 0 else 0,
            ])
        else:
            row["days_since_first_usage"] = 999
            row["days_since_last_usage"]  = 999
            row["usage_recency_score"]    = 0
            row["action_trend_ratio"]     = 0
            row["module_diversity"]       = 0
        
        all_rows.append(row)
    
    return pd.DataFrame(all_rows).set_index("ID")

In [148]:
cust_cutoffs = {row["ID"]: row["CLOSEDATE"] for _, row in customers_df.iterrows()}
all_cust_ids = set(companies_df["ID"])
cust_usage_feats = build_usage_features(list(all_cust_ids), cust_cutoffs)
cust_usage_feats

,ACTIONS_CRM_CONTACTS_sum_7d,ACTIONS_CRM_CONTACTS_mean_7d,ACTIONS_CRM_COMPANIES_sum_7d,ACTIONS_CRM_COMPANIES_mean_7d,ACTIONS_CRM_DEALS_sum_7d,ACTIONS_CRM_DEALS_mean_7d,ACTIONS_EMAIL_sum_7d,ACTIONS_EMAIL_mean_7d,USERS_CRM_CONTACTS_sum_7d,USERS_CRM_CONTACTS_mean_7d,...,total_actions_60d,total_users_60d,total_actions_lifetime,total_users_lifetime,active_days_lifetime,days_since_first_usage,days_since_last_usage,usage_recency_score,action_trend_ratio,module_diversity
ID,,,,,,,,,,,,,,,,,,,,,
1,208,208.0,61,61.0,87,87.0,7,7.0,4,4.0,...,917,30,917,30,4,23,2,0.333333,917.000000,4
2,2,2.0,0,0.0,14,14.0,0,0.0,1,1.0,...,123,12,920,59,14,106,1,0.500000,0.333333,4
3,1407,1407.0,220,220.0,439,439.0,5,5.0,24,24.0,...,12461,399,35455,1213,31,218,1,0.500000,1.816271,4
4,1510,1510.0,2,2.0,38,38.0,0,0.0,11,11.0,...,7528,116,39917,557,53,406,7,0.125000,3.468249,3
5,17,17.0,18,18.0,27,27.0,0,0.0,4,4.0,...,806,94,1391,213,30,255,3,0.250000,1.123684,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5196,59,59.0,0,0.0,0,0.0,0,0.0,4,4.0,...,487,39,562,52,19,182,7,0.125000,1.076596,3
5197,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,...,2,1,7,5,12,301,7,0.125000,0.000000,3
5198,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0,...,0,0,0,0,0,999,999,0.000000,0.000000,0


In [149]:
cust_usage_feats.columns

Index(['ACTIONS_CRM_CONTACTS_sum_7d', 'ACTIONS_CRM_CONTACTS_mean_7d',
       'ACTIONS_CRM_COMPANIES_sum_7d', 'ACTIONS_CRM_COMPANIES_mean_7d',
       'ACTIONS_CRM_DEALS_sum_7d', 'ACTIONS_CRM_DEALS_mean_7d',
       'ACTIONS_EMAIL_sum_7d', 'ACTIONS_EMAIL_mean_7d',
       'USERS_CRM_CONTACTS_sum_7d', 'USERS_CRM_CONTACTS_mean_7d',
       'USERS_CRM_COMPANIES_sum_7d', 'USERS_CRM_COMPANIES_mean_7d',
       'USERS_CRM_DEALS_sum_7d', 'USERS_CRM_DEALS_mean_7d',
       'USERS_EMAIL_sum_7d', 'USERS_EMAIL_mean_7d', 'active_days_7d',
       'total_actions_7d', 'total_users_7d', 'ACTIONS_CRM_CONTACTS_sum_14d',
       'ACTIONS_CRM_CONTACTS_mean_14d', 'ACTIONS_CRM_COMPANIES_sum_14d',
       'ACTIONS_CRM_COMPANIES_mean_14d', 'ACTIONS_CRM_DEALS_sum_14d',
       'ACTIONS_CRM_DEALS_mean_14d', 'ACTIONS_EMAIL_sum_14d',
       'ACTIONS_EMAIL_mean_14d', 'USERS_CRM_CONTACTS_sum_14d',
       'USERS_CRM_CONTACTS_mean_14d', 'USERS_CRM_COMPANIES_sum_14d',
       'USERS_CRM_COMPANIES_mean_14d', 'USERS_CRM_DEALS_sum_